# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZiadYakout/FlyRank-Ai/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


### Method: Random Forest Classification

I will use a Random Forest classifier for this lane.

The decision I want to support is which content items should receive attention first. The model can combine several page-level signals and capture non-linear relationships between them.

Random Forest is a suitable first model because it can handle interactions between signals without requiring a linear relationship between every feature and the outcome. It also provides a useful comparison against the simple Week-4 rule baseline.

I will not treat the model as automatically deciding what to do with a page. Its output is decision-support: it estimates the likelihood of the defined observed outcome so that pages can be ranked for review.

The model will use only information available at the decision moment. Label-derived fields such as `trend_direction` and `trend_pct` are excluded because they would leak information about the outcome.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.inspection import permutation_importance

# Load the starter dataset
df = pd.read_csv("/content/content_refresh_anonymized (1).csv")

print("Dataset shape:", df.shape)

# Create the observed starter decline label.
# This is used for this modeling exercise, but the final warehouse
# work should use a properly time-separated future outcome.
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Declining base rate:",
      round(df["is_declining_label"].mean(), 4))

Dataset shape: (30000, 44)
Declining base rate: 0.5421


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I will use a **grouped-by-client split**.

The group is `client_id`, so content from the same client will not appear in both the training and test sets. This is more honest for the decision because pages belonging to the same client can share characteristics that would make a random row-level split look easier than the real problem.

I will use 75% of the clients for training and 25% for testing.

The test clients will be kept separate until the final comparison with the baseline.

This split is intended to test whether the model can generalize across clients rather than simply memorize client-specific patterns.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Five decision-time features.
features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "days_since_last_update"
]

target = "is_declining_label"
group = "client_id"

# Keep only the columns we need
model_df = df[
    features + [target, group, "content_id"]
].copy()

# Remove rows where the target is unavailable
model_df = model_df.dropna(subset=[target])

# Replace missing feature values with median values calculated
# from the training data later.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df[target],
        groups=model_df[group]
    )
)

train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

print("Training rows:", len(train))
print("Testing rows:", len(test))

print("\nTraining clients:", train[group].nunique())
print("Testing clients:", test[group].nunique())

print("\nClients shared between train/test:",
      len(set(train[group]) & set(test[group])))

Training rows: 22885
Testing rows: 7115

Training clients: 24
Testing clients: 8

Clients shared between train/test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I will train the Random Forest using the same observed target and the same client-grouped split used for evaluation.

The model's probability of decline will be used as its ranking score.

I will compare the model against the Week-4 baseline using the same test rows and the same ranking metric.

The primary metric is **Precision@K**, because the practical use case is a limited review queue: the reviewer can only inspect a certain number of pages first.

I will also report ROC-AUC as a secondary diagnostic metric.

The baseline will use the same rule from Week 4 rather than being re-tuned after seeing the model results.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ---------------------------------------------------------
# Prepare X and y
# ---------------------------------------------------------

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

# ---------------------------------------------------------
# Model
# ---------------------------------------------------------

rf_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )
    )
])

rf_model.fit(X_train, y_train)

# Model probability
model_probability = rf_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# BASELINE SCORE
# Same Week-4 rule:
# stale + visible
# ---------------------------------------------------------

baseline_test = test.copy()

stale = (
    baseline_test["days_since_last_update"] >= 180
).astype(int)

visible = (
    baseline_test["impressions_90d"] >= 500
).astype(int)

baseline_test["baseline_score"] = (
    stale
    * visible
    * baseline_test["impressions_90d"]
)

# ---------------------------------------------------------
# Precision@K
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()


# Use 10% of the test set as the review queue.
K = max(1, int(len(test) * 0.10))

model_p_at_k = precision_at_k(
    y_test,
    model_probability,
    K
)

baseline_p_at_k = precision_at_k(
    y_test,
    baseline_test["baseline_score"],
    K
)

# Secondary metric
model_auc = roc_auc_score(
    y_test,
    model_probability
)

# Baseline AUC is only meaningful if there is variation in scores
if baseline_test["baseline_score"].nunique() > 1:
    baseline_auc = roc_auc_score(
        y_test,
        baseline_test["baseline_score"]
    )
else:
    baseline_auc = np.nan


# ---------------------------------------------------------
# MODEL VS BASELINE TABLE
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@K": [
        baseline_p_at_k,
        model_p_at_k
    ],
    "ROC-AUC": [
        baseline_auc,
        model_auc
    ],
    "K": [
        K,
        K
    ]
})

comparison

,method,Precision@K,ROC-AUC,K
0,Week-4 baseline,0.528833,0.500408,711
1,Random Forest,0.722925,0.625977,711


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


The comparison table above is the main test of whether the Random Forest adds value beyond the simple baseline.

If the Random Forest has higher Precision@K on the held-out clients, it provides evidence that combining several signals may improve the review queue.

If it does not outperform the baseline, that is also an important result. The simpler rule may be sufficient, or the available signals may not contain enough additional information for this model to improve the decision.

I will therefore prefer the simpler baseline unless the model shows a meaningful and reproducible improvement on the held-out clients.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Calculate permutation importance on the held-out test clients.
perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

print("Permutation importance:")
display(importance)

Permutation importance:


,feature,importance_mean,importance_std
0,impressions_90d,0.080133,0.004267
3,avg_position,0.046539,0.002737
1,clicks_90d,0.045461,0.002713
2,sessions_90d,0.013938,0.001375
4,days_since_last_update,0.002943,0.001961


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.